# Research-grounded BoC forecast

Inspect the documents visible at one historical forecast origin and the exact agent prompt. The live model call is opt-in so opening or running the notebook does not accidentally incur cost.


To run this code via CLI, execute the following code:
1. `uv run jupyter lab implementations/boc_rate_decisions/04_research_grounded_forecast.ipynb`
2. `uv run python implementations/boc_rate_decisions/run_research_agent.py --live`
3. `uv run python implementations/boc_rate_decisions/run_research_agent.py --origin 2024-08-07 --max-documents 2 --max-chars-per-document 4000 --live`

In [1]:
import json
from datetime import datetime
from pathlib import Path

import yaml
from IPython.display import JSON, Markdown, display
from aieng.forecasting.evaluation import BacktestSpec
from boc_rate_decisions.analyst_agent import (
    BoCDecisionPromptBuilder,
    build_boc_research_predictor,
)
from boc_rate_decisions.data import build_boc_service
from boc_rate_decisions.research import DEFAULT_RESEARCH_SOURCES, format_research_evidence


## Configuration

The default origin is 28 days before the June 5, 2024 decision. Set `RUN_LIVE = True` only when model credentials are configured and you intend to make a billed model call.


In [13]:
def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "implementations/boc_rate_decisions").is_dir() and (directory / "scripts").is_dir():
            return directory
    raise FileNotFoundError(f"Could not locate the repository root from {candidate}")

REPO_ROOT = find_repo_root()
ORIGIN = datetime(2024, 5, 8)
REPORTS_DIR = REPO_ROOT / "data/reports/boc_press_releases"
SPEC_PATH = REPO_ROOT / "implementations/boc_rate_decisions/specs/boc_rate_direction_smoke.yaml"
MAX_DOCUMENTS = 3
MAX_CHARS_PER_DOCUMENT = 6_000
RUN_LIVE = True

print(f"Repository root: {REPO_ROOT}")
print(f"Reports directory: {REPORTS_DIR}")
if not REPORTS_DIR.is_dir():
    raise FileNotFoundError(
        f"BoC release cache not found at {REPORTS_DIR}. "
        "Run from the repository root: uv run python scripts/fetch_boc_press_releases.py --year 2024"
    )

Repository root: /home/coder/agentic-forecasting
Reports directory: /home/coder/agentic-forecasting/data/reports/boc_press_releases


In [14]:
print("Kernel working directory:", Path.cwd())
print("Resolved reports directory:", REPORTS_DIR)
print("Directory exists:", REPORTS_DIR.is_dir())
print("JSON files:", len(list(REPORTS_DIR.glob("*.json"))))

Kernel working directory: /home/coder/agentic-forecasting/implementations/boc_rate_decisions
Resolved reports directory: /home/coder/agentic-forecasting/data/reports/boc_press_releases
Directory exists: True
JSON files: 140


In [15]:
if not REPORTS_DIR.exists():
    raise FileNotFoundError(
        f"BoC release cache not found at {REPORTS_DIR.resolve()}. "
        "Run: uv run python scripts/fetch_boc_press_releases.py --year 2024"
    )

## Load the cutoff-scoped data context


In [16]:
with SPEC_PATH.open(encoding="utf-8") as file:
    task = BacktestSpec.model_validate(yaml.safe_load(file)).task

service = build_boc_service(reports_dir=REPORTS_DIR)
context = service.context(ORIGIN)
print(f"Origin: {ORIGIN.date()} | Task: {task.task_id}")

Origin: 2024-05-08 | Task: boc_rate_direction_next_meeting


## Cutoff-visible evidence

Every displayed document must have a publication date on or before the origin.


In [17]:
evidence = format_research_evidence(
    context,
    sources=DEFAULT_RESEARCH_SOURCES,
    max_documents=MAX_DOCUMENTS,
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)
assert all(item["publication_date"] <= ORIGIN.date().isoformat() for item in evidence)
display(JSON(evidence, expanded=False))

<IPython.core.display.JSON object>

## Generated prompt

This is the exact JSON prompt that will be sent to the research-grounded analyst. Inspect it before enabling the model call.


In [18]:
builder = BoCDecisionPromptBuilder(
    document_sources=DEFAULT_RESEARCH_SOURCES,
    max_documents=MAX_DOCUMENTS,
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)
prompt = builder(task=task, context=context)
display(JSON(json.loads(prompt), expanded=False))

<IPython.core.display.JSON object>

## Live model prediction

Change `RUN_LIVE` to `True` in the configuration cell to invoke the configured model. The result is rendered as Markdown with probabilities, rationale, and key signals.


In [19]:
if RUN_LIVE:
    predictor = build_boc_research_predictor(
        max_documents=MAX_DOCUMENTS,
        max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
    )
    predictions = predictor.predict(task, context)
    if not predictions:
        raise RuntimeError("The research agent returned no prediction.")

    prediction = predictions[0]
    probabilities = prediction.payload.probabilities
    rationale = prediction.metadata.get("rationale", "No rationale returned.")
    signals = prediction.metadata.get("key_signals", [])
    rows = "\n".join(f"| {label} | {probability:.1%} |" for label, probability in probabilities.items())
    signal_text = "\n".join(f"- {signal}" for signal in signals) or "- None returned"
    display(Markdown(
        f"### Forecast for {prediction.forecast_date:%Y-%m-%d}\n\n"
        f"| Decision | Probability |\n|---|---:|\n{rows}\n\n"
        f"**Rationale:** {rationale}\n\n**Key signals:**\n{signal_text}"
    ))
else:
    display(Markdown("> Live prediction skipped. Set `RUN_LIVE = True` in the configuration cell to call the model."))

/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


### Forecast for 2024-06-05

| Decision | Probability |
|---|---:|
| cut | 45.0% |
| hold | 53.0% |
| hike | 2.0% |

**Rationale:** As of May 8, 2024, the Bank of Canada has held the overnight rate at 5.0% since July 2023. The macro data shows clear signs of progress: inflation has decelerated to 2.8% (as of the April 10 announcement), the economy is in excess supply, and the labor market is softening (unemployment at 6.1% in March). Critically, the Governing Council stated in the April 10 press release that 'CPI and core inflation have eased further' and that they are looking for evidence that this downward momentum is sustained. The market is actively pricing in the start of an easing cycle for the mid-year meetings. While the Bank's mandate requires caution, the 'hold' probability remains higher due to the institution's hallmark gradualism and reluctance to surprise, requiring them to wait for the confirmation of data trends in the weeks preceding June 5. A cut in June is highly plausible given the documented momentum, but a hold remains the slightly more probable base case.

**Key signals:**
- 2024-04-10_en press release: 'CPI and core inflation have eased further... The Council will be looking for evidence that this downward momentum is sustained.'
- Rising unemployment momentum (6.1% in March 2024).
- Inflation gap narrowing (CPI inflation 2.8% vs 2% target).
- Yield spread significantly negative, signaling market expectation of near-term rate cuts.